# EEG_35 — Subject-Invariant Dynamic Hypergraph Network (SI-DHGN)

**Contesto (da EEG_34).** L'analisi informazione-teorica ha mostrato che il segnale EEG
contiene informazione semantica sufficiente per un'accuracy teorica del **47–79%** (Fano su MINE),
mentre il nostro modello cross-subject si ferma a **~26%**. Siamo **model-limited, non data-limited**:
il collo di bottiglia è la **generalizzazione inter-soggetto** (ε²(soggetto)=0.85), non l'assenza di
informazione. L'identità del soggetto *maschera* la firma semantica.

**Idea.** Non serve un HGNN *più grande* — quello overfitterebbe sull'identità del soggetto. Serve un
HGNN che **combatte attivamente** l'identità, lasciando emergere il contenuto lessicale. Costruiamo un
unico modello sopra il backbone **DHSLP** (ipergrafo dinamico, iperedge apprese) che integra tre leve
indipendenti contro la variabilità inter-soggetto, ciascuna anche regolarizzante:

| Leva | Cosa fa | Riferimento |
|------|---------|-------------|
| **Instance Norm + Subject-Mean Subtraction** | rimuove offset ampiezza e DC pattern del soggetto | Bomatter 2024 / EEG_29 |
| **GRL adversariale sul soggetto** (DANN) | l'encoder è penalizzato se l'embedding è subject-identificabile | Ganin 2016 / EEG_26 |
| **Supervised Contrastive (SupCon)** | avvicina trial della stessa parola *tra* soggetti diversi | Khosla 2020 / Shen 2022 |

**Anti-overfitting (esplicito).** AdamW + weight decay, **DropEdge** sulle iperedge soft, label smoothing,
dropout alto, cosine LR con warm-up, early stopping su val bACC. Ogni leva anti-soggetto è *anche* una
forma di regolarizzazione: spingono accuracy e generalizzazione nella stessa direzione.

**Disegno sperimentale.** Le leve sono **flag ablabili**. Confrontiamo il baseline DHSLP puro contro il
SI-DHGN completo e (opzionale) le ablation per isolare il contributo di ciascuna leva. Tracking W&B
obbligatorio (CLAUDE.md §8.1). Split subject-independent standard: TRAIN 0–49, VAL 50–59, TEST 60–73.


In [ ]:
# ====================== §0 — Setup & Config ======================
import json, logging, re, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from tqdm.auto import tqdm

try:
    import wandb
    _HAS_WANDB = True
except Exception:
    _HAS_WANDB = False

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg35')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---- DATI ----
N_CHANNELS  = 61
N_SAMPLES   = 384
N_CLASSES   = 4
DATA_METRIC = 'abs_pcc'
CLUSTER_SCHEME = 'concr4'
CLUSTER_NAMES  = ['CONCR', 'AZIONE', 'STATO', 'ASTR']

# Split subject-independent standard (identico a EEG_13/26/29/34)
SUBJ_TRAIN = list(range(0, 50))
SUBJ_VAL   = list(range(50, 60))
SUBJ_TEST  = list(range(60, 74))

# QUICK=True per smoke-test locale veloce (sottoinsieme soggetti, poche epoche)
QUICK = False
if QUICK:
    SUBJ_TRAIN, SUBJ_VAL, SUBJ_TEST = list(range(0,12)), list(range(50,54)), list(range(60,64))

# ---- ARCHITETTURA (backbone DHSLP, best config EEG_13) ----
K_WINDOWS = 8; T_WIN = N_SAMPLES // K_WINDOWS
N_EDGES   = 16      # n. iperedge apprese
D_MODEL   = 64      # dim proiezione nodo / spazio iperedge
D_ENC     = 128     # dim embedding latente (hidden HGNN)
N_LAYERS  = 2
DROPOUT   = 0.4     # dropout alto = regolarizzazione
DROPEDGE  = 0.10    # prob. di azzerare incidenze nodo-iperedge (DropEdge)

# ---- OTTIMIZZAZIONE ----
LR              = 1e-3
WEIGHT_DECAY    = 1e-3      # AdamW: regolarizzazione L2 disaccoppiata
BATCH_SIZE      = 64
MAX_EPOCHS      = 60 if not QUICK else 6
PATIENCE        = 12
WARMUP_EPOCHS   = 5
LABEL_SMOOTHING = 0.1
SEEDS           = [42, 123, 7] if not QUICK else [42]

# ---- LEVE ANTI-SOGGETTO (pesi loss / schedule) ----
LAMBDA_GRL_MAX  = 0.5       # peso massimo domain-adversarial (schedule Ganin)
SUPCON_WEIGHT   = 0.3       # peso supervised contrastive
SUPCON_TAU      = 0.1       # temperatura SupCon

WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'
USE_WANDB     = _HAS_WANDB

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k, v in json.loads(
    (project_root/'configs'/'label_schemes'/'labelid2cluster_concr4.json').read_text()).items()}

log.info(f'device={device} | TRAIN={len(SUBJ_TRAIN)} VAL={len(SUBJ_VAL)} TEST={len(SUBJ_TEST)} | QUICK={QUICK}')


## §1 — Dati: loader subject-independent

Sorgente: `data/hypergraphs_pruned_abs_pcc/PXXX_SYYY/trial_*.pt` (server-side), ognuno con `x` (61,384)
e `y` (word label 0–109). Mappiamo a cluster **concr4**. Ogni trial subisce **instance norm** e,
opzionalmente, **Subject-Mean Subtraction** (μ del soggetto sottratta). Il dataset restituisce anche il
**domain label** (indice soggetto) per la domain loss del GRL.


In [ ]:
# ====================== §1.1 — Medie per soggetto (SMS) ======================
def compute_subject_means(subj_ids, metric=DATA_METRIC):
    '''mu_s = media dei trial instance-norm del soggetto s -> dict sid -> (61,384).'''
    root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
    accum = defaultdict(list)
    for p in sorted(root.rglob('trial_*.pt')):
        m = _PAT.match(p.parent.name)
        if not m: continue
        sid = int(m.group(1))
        if sid not in subj_ids: continue
        d = torch.load(p, weights_only=False)
        x = d['x'].float()
        x = (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + 1e-6)
        accum[sid].append(x)
    means = {sid: torch.stack(t).mean(0) for sid, t in accum.items()}
    log.info(f'mu calcolate per {len(means)} soggetti')
    return means

DATA_ROOT = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'
DATA_AVAILABLE = DATA_ROOT.exists() and any(DATA_ROOT.rglob('trial_*.pt'))
log.info(f'DATA_AVAILABLE={DATA_AVAILABLE} ({DATA_ROOT})')

subj_means = compute_subject_means(SUBJ_TRAIN + SUBJ_VAL + SUBJ_TEST) if DATA_AVAILABLE else {}


In [ ]:
# ====================== §1.2 — Dataset & Loader ======================
class EEGDataset(Dataset):
    '''Restituisce (x(61,384), cluster_label, domain_idx). domain_idx = indice soggetto 0-based nello split.'''
    def __init__(self, subj_ids, subject_means=None, use_sms=False, metric=DATA_METRIC):
        root = project_root / 'data' / f'hypergraphs_pruned_{metric}'
        self.paths, self.labels, self.domains = [], [], []
        self.subject_means, self.use_sms = subject_means, use_sms
        subj2dom = {sid: i for i, sid in enumerate(sorted(subj_ids))}
        self._sids = []
        for p in sorted(root.rglob('trial_*.pt')):
            m = _PAT.match(p.parent.name)
            if not m: continue
            sid = int(m.group(1))
            if sid not in subj_ids: continue
            d = torch.load(p, weights_only=False)
            yw = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
            c = label2cluster.get(yw)
            if c is None: continue
            self.paths.append(p); self.labels.append(c)
            self.domains.append(subj2dom[sid]); self._sids.append(sid)
        self.n_domains = len(subj2dom)
        log.info(f'  {len(self.paths)} trial | {len(subj_ids)} soggetti | SMS={use_sms}')

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        d = torch.load(self.paths[idx], weights_only=False)
        x = d['x'].float()
        x = (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + 1e-6)   # instance norm sempre
        if self.use_sms and self.subject_means is not None:
            sid = self._sids[idx]
            if sid in self.subject_means:
                x = x - self.subject_means[sid]
        return (x,
                torch.tensor(self.labels[idx],  dtype=torch.long),
                torch.tensor(self.domains[idx], dtype=torch.long))


def make_loaders(use_sms):
    tr = EEGDataset(SUBJ_TRAIN, subj_means, use_sms=use_sms)
    va = EEGDataset(SUBJ_VAL,   subj_means, use_sms=use_sms)
    te = EEGDataset(SUBJ_TEST,  subj_means, use_sms=use_sms)
    labels = np.array(tr.labels)
    counts = np.bincount(labels, minlength=N_CLASSES)
    w = torch.tensor(1.0 / np.clip(counts[labels], 1, None), dtype=torch.float)
    sampler = WeightedRandomSampler(w, len(w), replacement=True)
    kw = dict(num_workers=2, pin_memory=True)
    return (DataLoader(tr, BATCH_SIZE, sampler=sampler, drop_last=True, **kw),
            DataLoader(va, BATCH_SIZE, shuffle=False, **kw),
            DataLoader(te, BATCH_SIZE, shuffle=False, **kw),
            tr.n_domains)


## §2 — Architettura SI-DHGN

**Backbone DHSLP.** Per ognuna delle K=8 finestre temporali: proiezione nodo `Linear(T_win→d_model)` +
positional encoding, poi le iperedge **dinamiche** `H_k = softmax(feat · Eᵀ)` (E apprese), e N_LAYERS
di convoluzione ipergrafica HGNN. Embedding finale = media sulle finestre → `z ∈ ℝ^{D_ENC}`.

**DropEdge.** In training azzeriamo con prob. `DROPEDGE` le incidenze nodo-iperedge di `H_k` e
rinormalizziamo: regolarizza l'ipergrafo, evita che il modello si appoggi su poche iperedge.

**Tre teste sopra l'encoder:**
- *Task head* → 4 classi concr4 (loss cross-entropy con label smoothing).
- *Domain head* preceduta da **GRL** → n soggetti del train (loss adversarial: l'encoder è spinto a
  rendere l'embedding **non** subject-identificabile).
- *Projection head* → spazio L2-normalizzato per **SupCon** (avvicina stessi-cluster, allontana diversi).


In [ ]:
# ====================== §2.1 — GRL, HGNNConv, DropEdge ======================
class GradReverseFn(Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.clone()
    @staticmethod
    def backward(ctx, g):
        return -ctx.lam * g, None

class GRL(nn.Module):
    def forward(self, x, lam=1.0): return GradReverseFn.apply(x, lam)

def grl_lambda(step, total, lam_max):
    '''Schedule di Ganin 2016: 0 -> lam_max, sigmoidale.'''
    p = step / max(total, 1)
    return float(lam_max * (2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0))

class HGNNConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_ch, out_ch))
        self.bias   = nn.Parameter(torch.zeros(out_ch))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, X, H):
        d_v = H.sum(2).clamp(min=1e-6); d_e = H.sum(1).clamp(min=1e-6)
        Dv  = (1.0/d_v.sqrt()).unsqueeze(-1); De = (1.0/d_e).unsqueeze(1)
        out = Dv * (X @ self.weight)
        out = torch.bmm(H.transpose(1,2), out)
        out = De.transpose(1,2) * out
        out = torch.bmm(H, out)
        return Dv * out + self.bias

def drop_edge(H, p, training):
    '''Azzera con prob. p le incidenze nodo-iperedge di H (B,N,E) e rinormalizza su E.'''
    if not training or p <= 0:
        return H
    mask = (torch.rand_like(H) > p).float()
    H = H * mask
    H = H / (H.sum(dim=2, keepdim=True) + 1e-6)   # rinormalizza la softmax sui rami superstiti
    return H


In [ ]:
# ====================== §2.2 — Modello SI-DHGN ======================
class DHSLPEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.E       = nn.Parameter(torch.randn(N_EDGES, D_MODEL) * 0.01)
        self.pos_enc = nn.Parameter(torch.randn(N_CHANNELS, D_MODEL) * 0.01)
        self.node_proj = nn.Sequential(nn.Linear(T_WIN, D_MODEL), nn.LayerNorm(D_MODEL), nn.ELU())
        dims = [D_MODEL] + [D_ENC] * N_LAYERS
        self.convs = nn.ModuleList([HGNNConv(dims[i], dims[i+1]) for i in range(N_LAYERS)])
        self.bns   = nn.ModuleList([nn.BatchNorm1d(D_ENC) for _ in range(N_LAYERS)])
        self.drop  = nn.Dropout(DROPOUT)
    def forward(self, x):
        B, N, _ = x.shape
        outs = []
        for k in range(K_WINDOWS):
            x_k  = x[:, :, k*T_WIN:(k+1)*T_WIN]
            feat = self.node_proj(x_k) + self.pos_enc
            H_k  = torch.softmax(torch.matmul(feat, self.E.T) / D_MODEL**0.5, dim=2)
            H_k  = drop_edge(H_k, DROPEDGE, self.training)
            out  = feat
            for conv, bn in zip(self.convs, self.bns):
                out = conv(out, H_k)
                out = bn(out.reshape(B*N, -1)).reshape(B, N, -1)
                out = F.relu(out); out = self.drop(out)
            outs.append(out.mean(1))
        return torch.stack(outs, 1).mean(1)   # (B, D_ENC)

class SIDHGN(nn.Module):
    '''Subject-Invariant Dynamic Hypergraph Net. Flag: use_grl, use_supcon attivano le teste extra.'''
    def __init__(self, n_domains, use_grl=True, use_supcon=True, proj_dim=64):
        super().__init__()
        self.use_grl, self.use_supcon = use_grl, use_supcon
        self.encoder = DHSLPEncoder()
        self.task_clf = nn.Sequential(nn.Linear(D_ENC, D_ENC//2), nn.ReLU(),
                                      nn.Dropout(DROPOUT), nn.Linear(D_ENC//2, N_CLASSES))
        if use_grl:
            self.grl = GRL()
            self.domain_clf = nn.Sequential(nn.Linear(D_ENC, D_ENC//2), nn.ReLU(),
                                            nn.Dropout(DROPOUT), nn.Linear(D_ENC//2, n_domains))
        if use_supcon:
            self.proj = nn.Sequential(nn.Linear(D_ENC, D_ENC), nn.ReLU(), nn.Linear(D_ENC, proj_dim))
    def forward(self, x, lam=1.0):
        z = self.encoder(x)
        task = self.task_clf(z)
        dom  = self.domain_clf(self.grl(z, lam)) if self.use_grl else None
        emb  = F.normalize(self.proj(z), dim=1) if self.use_supcon else None
        return task, dom, emb

def supcon_loss(emb, labels, tau=SUPCON_TAU):
    '''Supervised Contrastive (Khosla 2020). emb L2-norm (B,d). Positivi = stesso cluster.'''
    B = emb.shape[0]
    sim = emb @ emb.T / tau
    sim = sim - sim.max(dim=1, keepdim=True).values.detach()
    eye = torch.eye(B, device=emb.device, dtype=torch.bool)
    same = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~eye
    exp = torch.exp(sim) * (~eye).float()
    log_prob = sim - torch.log(exp.sum(1, keepdim=True) + 1e-9)
    pos_cnt = same.sum(1)
    valid = pos_cnt > 0
    if valid.sum() == 0:
        return emb.sum() * 0.0
    mean_pos = (log_prob * same.float()).sum(1)[valid] / pos_cnt[valid]
    return -mean_pos.mean()

# Sanity
if True:
    _m = SIDHGN(n_domains=23).to(device)
    _t,_d,_e = _m(torch.randn(4,N_CHANNELS,N_SAMPLES).to(device), lam=0.5)
    assert _t.shape==(4,N_CLASSES) and _d.shape==(4,23) and _e.shape[0]==4
    log.info(f'SIDHGN OK — {sum(p.numel() for p in _m.parameters() if p.requires_grad):,} param')
    del _m,_t,_d,_e


## §3 — Training

Loss combinata: `L = L_task + λ(t)·L_domain + w_supcon·L_supcon`.

- `L_task`: cross-entropy con label smoothing (regolarizza le label).
- `L_domain`: cross-entropy sul soggetto, gradiente **invertito** dal GRL con λ che cresce 0→0.5
  (schedule di Ganin: all'inizio non disturba il task, poi forza l'invarianza).
- `L_supcon`: supervised contrastive sul batch.

Ottimizzatore **AdamW** (weight decay disaccoppiato) + **cosine LR** con warm-up lineare. Early stopping
sulla **val bACC**. Selezione modello = best val bACC.


In [ ]:
# ====================== §3.1 — Loop di training ======================
def set_seed(s):
    torch.manual_seed(s); np.random.seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for x, y, _ in loader:
        x = x.to(device)
        logits, _, _ = model(x, lam=0.0)
        ps.append(logits.argmax(1).cpu()); ys.append(y)
    y = torch.cat(ys).numpy(); p = torch.cat(ps).numpy()
    return balanced_accuracy_score(y, p), y, p

def train_model(name, use_sms, use_grl, use_supcon, seed, run=None):
    set_seed(seed)
    tr_loader, va_loader, te_loader, n_dom = make_loaders(use_sms)
    model = SIDHGN(n_dom, use_grl=use_grl, use_supcon=use_supcon).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    ce  = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    ce_dom = nn.CrossEntropyLoss()

    total_steps = MAX_EPOCHS * max(len(tr_loader), 1)
    def lr_at(step):
        ep = step / max(len(tr_loader), 1)
        if ep < WARMUP_EPOCHS:
            return ep / max(WARMUP_EPOCHS, 1e-9)
        prog = (ep - WARMUP_EPOCHS) / max(MAX_EPOCHS - WARMUP_EPOCHS, 1e-9)
        return 0.5 * (1.0 + math.cos(math.pi * min(prog, 1.0)))

    best_va, best_state, bad, step = -1.0, None, 0, 0
    for epoch in range(MAX_EPOCHS):
        model.train(); agg = defaultdict(float); nb = 0
        for x, y, dom in tr_loader:
            x, y, dom = x.to(device), y.to(device), dom.to(device)
            for g in opt.param_groups: g['lr'] = LR * lr_at(step)
            lam = grl_lambda(step, total_steps, LAMBDA_GRL_MAX) if use_grl else 0.0
            logits, dlog, emb = model(x, lam=lam)
            loss = ce(logits, y)
            l_dom = ce_dom(dlog, dom) if use_grl else torch.zeros((), device=device)
            l_sc  = supcon_loss(emb, y) if use_supcon else torch.zeros((), device=device)
            loss = loss + (l_dom if use_grl else 0.0) + (SUPCON_WEIGHT * l_sc if use_supcon else 0.0)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            agg['task'] += ce(logits, y).item(); agg['dom'] += float(l_dom); agg['sc'] += float(l_sc)
            nb += 1; step += 1
        va_bacc, _, _ = evaluate(model, va_loader)
        if run is not None:
            run.log({'train/task_loss': agg['task']/max(nb,1), 'train/dom_loss': agg['dom']/max(nb,1),
                     'train/supcon_loss': agg['sc']/max(nb,1), 'val/bacc': va_bacc,
                     'lam_grl': lam, 'lr': LR*lr_at(step), 'epoch': epoch})
        if va_bacc > best_va:
            best_va, best_state, bad = va_bacc, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
        log.info(f'[{name} s{seed}] ep{epoch:02d} val_bACC={va_bacc:.4f} (best={best_va:.4f}) '
                 f'task={agg["task"]/max(nb,1):.3f} dom={agg["dom"]/max(nb,1):.3f} sc={agg["sc"]/max(nb,1):.3f}')
        if bad >= PATIENCE:
            log.info(f'[{name} s{seed}] early stop @ep{epoch}'); break

    if best_state is not None: model.load_state_dict(best_state)
    te_bacc, y_te, p_te = evaluate(model, te_loader)
    log.info(f'[{name} s{seed}] >>> val_bACC={best_va:.4f}  test_bACC={te_bacc:.4f}')
    return dict(val_bacc=best_va, test_bacc=te_bacc, y_te=y_te, p_te=p_te)


## §4 — Esperimenti

Confrontiamo configurazioni che isolano il contributo di ciascuna leva. Ognuna gira su `SEEDS` multipli
e logga una run W&B separata (`eeg35_{config}_concr4`). Per un primo giro veloce si possono disattivare
le ablation lasciando solo `baseline` e `full`.


In [ ]:
# ====================== §4.1 — Griglia configurazioni ======================
# (nome, use_sms, use_grl, use_supcon)
CONFIGS = [
    ('baseline_dhslp', False, False, False),   # DHSLP puro (riferimento)
    ('sms',            True,  False, False),    # + subject-mean subtraction
    ('grl',            True,  True,  False),    # + adversarial soggetto
    ('full_sidhgn',    True,  True,  True),     # + supervised contrastive (modello completo)
]
RUN_ABLATIONS = True
if not RUN_ABLATIONS:
    CONFIGS = [CONFIGS[0], CONFIGS[-1]]

results = {}
if DATA_AVAILABLE:
    for (name, sms, grl, sc) in CONFIGS:
        seed_res = []
        for seed in SEEDS:
            run = None
            if USE_WANDB:
                try:
                    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                                     name=f'eeg35_{name}_{CLUSTER_SCHEME}_s{seed}',
                                     group='eeg35_subject_invariant_hypergraph',
                                     config=dict(notebook='EEG_35', model=name, use_sms=sms, use_grl=grl,
                                                 use_supcon=sc, n_classes=N_CLASSES, cluster_scheme=CLUSTER_SCHEME,
                                                 lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH_SIZE,
                                                 max_epochs=MAX_EPOCHS, k_windows=K_WINDOWS, n_edges=N_EDGES,
                                                 d_enc=D_ENC, dropout=DROPOUT, dropedge=DROPEDGE,
                                                 lambda_grl_max=LAMBDA_GRL_MAX, supcon_weight=SUPCON_WEIGHT,
                                                 n_train_subj=len(SUBJ_TRAIN), seed=seed),
                                     reinit='finish_previous')
                except Exception as e:
                    log.warning(f'W&B init fallita ({e})'); run = None
            r = train_model(name, sms, grl, sc, seed, run=run)
            if run is not None:
                run.summary['val_bacc']  = r['val_bacc']
                run.summary['test_bacc'] = r['test_bacc']
                try:
                    run.log({'confusion_matrix': wandb.plot.confusion_matrix(
                        y_true=r['y_te'].tolist(), preds=r['p_te'].tolist(), class_names=CLUSTER_NAMES)})
                except Exception: pass
                run.finish()
            seed_res.append(r)
        results[name] = seed_res
        vb = np.array([r['val_bacc'] for r in seed_res]); tb = np.array([r['test_bacc'] for r in seed_res])
        log.info(f'=== {name}: val {vb.mean():.4f}±{vb.std():.4f} | test {tb.mean():.4f}±{tb.std():.4f} ===')
else:
    log.warning('[INFO] Dati non disponibili in locale: esegui questa cella sul server (env daniele_311).')


## §5 — Risultati: confronto configurazioni

In [ ]:
# ====================== §5.1 — Tabella + plot ======================
if results:
    import pandas as pd
    rows = []
    for name, seed_res in results.items():
        vb = np.array([r['val_bacc'] for r in seed_res]); tb = np.array([r['test_bacc'] for r in seed_res])
        rows.append(dict(config=name, val_bacc=vb.mean(), val_std=vb.std(),
                         test_bacc=tb.mean(), test_std=tb.std()))
    res_df = pd.DataFrame(rows)
    base = res_df.loc[res_df.config=='baseline_dhslp','test_bacc']
    base = float(base.iloc[0]) if len(base) else np.nan
    res_df['delta_vs_base'] = res_df['test_bacc'] - base
    print(res_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(max(7, 1.7*len(res_df)+3), 4.8))
    x = np.arange(len(res_df)); w = 0.36
    ax.bar(x - w/2, res_df.val_bacc,  w, yerr=res_df.val_std,  capsize=3, label='val',  color='#A8C8E8', edgecolor='k', lw=.5)
    ax.bar(x + w/2, res_df.test_bacc, w, yerr=res_df.test_std, capsize=3, label='test', color='#E8A8A8', edgecolor='k', lw=.5)
    ax.axhline(1/N_CLASSES, ls='--', color='gray', lw=1, label=f'chance ({1/N_CLASSES:.0%})')
    ax.set_xticks(x); ax.set_xticklabels(res_df.config, rotation=18, ha='right', fontsize=9)
    ax.set_ylabel('Balanced Accuracy (concr4)')
    ax.set_title('EEG_35 — SI-DHGN: ablation leve anti-soggetto (subject-independent)')
    ax.legend(fontsize=9)
    for xi, v in zip(x, res_df.test_bacc): ax.text(xi+w/2, v+0.004, f'{v:.3f}', ha='center', fontsize=7)
    plt.tight_layout()
    out = FIG_DIR / 'eeg35_sidhgn_ablation.png'
    plt.savefig(out, dpi=150, bbox_inches='tight'); log.info(f'Figura: {out}')
    plt.show()
else:
    print('Nessun risultato (dati non disponibili in locale). Esegui sul server.')


## §6 — Come leggere i risultati

| Esito | Interpretazione |
|-------|-----------------|
| `full_sidhgn` >> `baseline_dhslp` | Le leve anti-soggetto sbloccano informazione semantica mascherata dall'identità → conferma EEG_34 (model-limited) e indica la **direzione corretta** per la tesi. |
| `grl` o `sms` portano il guadagno, `supcon` marginale | Il bottleneck è rimuovibile con domain-invariance esplicita; il contrastive aggiunge poco con questi dati. |
| Tutte le config ≈ baseline | Le leve testate non bastano: l'identità del soggetto è intrecciata alla firma semantica a un livello che richiede preprocessing/architettura diversi (test-time adaptation, più finestre, raw multi-banda). |
| `val >> test` ovunque | Overfitting residuo sul pool train: aumentare DROPOUT/DROPEDGE/weight_decay o ridurre D_ENC. |

**Note metodologiche da dichiarare in tesi.**
1. Split subject-independent rigoroso: i soggetti di test non compaiono mai in training (né per SMS, μ è per-soggetto).
2. Selezione modello su **val bACC**, mai su test.
3. Il GRL usa solo le label-soggetto del **train** (domain classifier su n soggetti train); a test λ=0.
4. Ogni leva è anche regolarizzazione → un guadagno non distingue *invarianza* da *minor overfitting*:
   l'ablation isola i due effetti.
